# Shri Yogavasishtha — Chandra OCR

**Runtime:** GPU (T4 or better)  
**Steps:**
1. Run all cells in order
2. Upload the 4 PDFs when prompted (or place them in Google Drive)
3. OCR output is saved to `/content/drive/MyDrive/yogavasishtha_ocr/`

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — switch runtime to GPU (Runtime > Change runtime type > T4 GPU)')

In [ ]:
# ── Cell 2: Install chandra-ocr ─────────────────────────────────────────────
!pip install -q chandra-ocr[hf]
print('chandra-ocr installed')

In [ ]:
# ── Cell 3: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/yogavasishtha_ocr'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output will be saved to: {OUTPUT_DIR}')

In [ ]:
# ── Cell 4: Upload PDFs ─────────────────────────────────────────────────────
# Option A: Upload from your computer
from google.colab import files

print('Select all 4 PDF files to upload (Shri-Yogavasishtha-1.pdf through 4.pdf)')
uploaded = files.upload()

# Move uploaded files to /content/
import shutil
pdf_paths = []
for filename in uploaded:
    dest = f'/content/{filename}'
    if not os.path.exists(dest):
        shutil.move(filename, dest)
    pdf_paths.append(dest)
    size_mb = os.path.getsize(dest) / 1e6
    print(f'  {filename}: {size_mb:.1f} MB')

pdf_paths.sort()
print(f'\nReady to OCR: {pdf_paths}')

In [ ]:
# ── Cell 4-alt: Use Google Drive PDFs instead of uploading ──────────────────
# Run this cell ONLY if you placed the PDFs in Google Drive already
# (skip Cell 4 above and run this one instead)

# CHANGE this path to wherever you put the PDFs in your Drive:
DRIVE_PDF_DIR = '/content/drive/MyDrive/Yogavasishtha_PDFs'

pdf_paths = sorted([
    os.path.join(DRIVE_PDF_DIR, f)
    for f in os.listdir(DRIVE_PDF_DIR)
    if f.endswith('.pdf')
])
print('PDFs found:')
for p in pdf_paths:
    print(f'  {p}')

In [ ]:
# ── Cell 5: Check page counts ───────────────────────────────────────────────
import pypdfium2 as pdfium

total_pages = 0
for pdf_path in pdf_paths:
    doc = pdfium.PdfDocument(pdf_path)
    pages = len(doc)
    doc.close()
    total_pages += pages
    print(f'{os.path.basename(pdf_path)}: {pages} pages')

print(f'\nTotal: {total_pages} pages')
# T4 GPU processes ~4-8 pages/min → estimate
est_min = total_pages / 6
print(f'Estimated time @ ~6 pages/min on T4: ~{est_min:.0f} min ({est_min/60:.1f} hrs)')

In [ ]:
# ── Cell 6: Run OCR on all 4 PDFs ───────────────────────────────────────────
import subprocess, sys, time

CHANDRA = sys.executable.replace('python3', 'chandra') if 'python3' in sys.executable else 'chandra'
# fallback
result = subprocess.run(['which', 'chandra'], capture_output=True, text=True)
if result.returncode == 0:
    CHANDRA = result.stdout.strip()

print(f'chandra path: {CHANDRA}\n')

for pdf_path in pdf_paths:
    name = os.path.splitext(os.path.basename(pdf_path))[0]
    out_dir = os.path.join(OUTPUT_DIR, name)
    os.makedirs(out_dir, exist_ok=True)

    # Check if already done
    done_files = [f for f in os.listdir(out_dir) if f.endswith('.md') or f.endswith('.txt')]
    if done_files:
        print(f'SKIP {name} — {len(done_files)} output files already exist')
        continue

    print(f'\n━━━ Starting OCR: {name} ━━━')
    t0 = time.time()

    cmd = [
        CHANDRA,
        pdf_path,
        out_dir,
        '--method', 'hf',
        '--no-headers-footers',   # skip headers/footers for cleaner output
        '--include-images',       # include image descriptions
        '--max-workers', '4',     # parallel page workers
    ]

    proc = subprocess.run(cmd, capture_output=False, text=True)

    elapsed = time.time() - t0
    status = 'DONE' if proc.returncode == 0 else f'ERROR (code {proc.returncode})'
    print(f'{status} — {elapsed/60:.1f} min')

print('\n✓ All OCR jobs complete')
print(f'Output in: {OUTPUT_DIR}')

In [ ]:
# ── Cell 7: Merge output into single files per part ─────────────────────────
for pdf_path in pdf_paths:
    name = os.path.splitext(os.path.basename(pdf_path))[0]
    out_dir = os.path.join(OUTPUT_DIR, name)
    merged_path = os.path.join(OUTPUT_DIR, f'{name}_merged.md')

    if not os.path.exists(out_dir):
        print(f'SKIP {name} — output dir not found')
        continue

    # Collect page markdown files in order
    md_files = sorted(
        [f for f in os.listdir(out_dir) if f.endswith('.md')],
        key=lambda x: int(''.join(filter(str.isdigit, x)) or '0')
    )

    with open(merged_path, 'w', encoding='utf-8') as out_f:
        out_f.write(f'# {name}\n\n')
        for md_file in md_files:
            page_num = ''.join(filter(str.isdigit, md_file))
            out_f.write(f'\n---\n<!-- page {page_num} -->\n\n')
            with open(os.path.join(out_dir, md_file), 'r', encoding='utf-8') as f:
                out_f.write(f.read())

    size_kb = os.path.getsize(merged_path) / 1024
    print(f'{name}: merged {len(md_files)} pages → {merged_path} ({size_kb:.0f} KB)')

print('\n✓ Merge complete')

In [ ]:
# ── Cell 8: Preview first page of Part 1 ────────────────────────────────────
merged_part1 = os.path.join(OUTPUT_DIR, 'Shri-Yogavasishtha-1_merged.md')
if os.path.exists(merged_part1):
    with open(merged_part1, 'r', encoding='utf-8') as f:
        content = f.read()
    print(content[:2000])
else:
    print('File not found — OCR may still be running')

In [ ]:
# ── Cell 9 (optional): Zip and download everything ──────────────────────────
import zipfile

zip_path = '/content/yogavasishtha_ocr.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(OUTPUT_DIR):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            arcname = os.path.relpath(filepath, OUTPUT_DIR)
            zf.write(filepath, arcname)

size_mb = os.path.getsize(zip_path) / 1e6
print(f'Zip created: {zip_path} ({size_mb:.1f} MB)')

# Download to your computer
files.download(zip_path)